# Lab 03: Prompt Engineering Patterns

## Overview
In this lab you will master prompt engineering techniques for Amazon Bedrock including zero-shot, few-shot, chain-of-thought, system prompts, structured output extraction, prompt templates, and prompt chaining. You will use the Converse API with Claude to explore how each technique changes model behavior and output quality.

## Learning Objectives
- Write effective zero-shot prompts and understand when they are sufficient
- Improve output quality with few-shot examples for classification and generation tasks
- Apply chain-of-thought (CoT) prompting to improve accuracy on reasoning tasks
- Use system prompts via the Converse API to control model persona and behavior
- Extract structured JSON output reliably from foundation models
- Chain multiple model calls together to decompose complex tasks into steps

## Exam Domain
**Building Generative AI Applications (30%)** — This lab covers Task 2.3 (prompt engineering techniques for optimizing model output quality and reliability).

## Architecture Diagram
![Lab 03 Architecture](../assets/diagrams/lab-03-prompt-engineering.png)

In [ ]:
%pip install boto3 --quiet

In [ ]:
import boto3, json, os

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock_runtime = session.client("bedrock-runtime")

MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"

def invoke(prompt, system=None, max_tokens=512, temperature=0.7):
    """Reusable helper to call Claude via the Converse API."""
    kwargs = {
        "modelId": MODEL_ID,
        "messages": [{"role": "user", "content": [{"text": prompt}]}],
        "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature}
    }
    if system:
        kwargs["system"] = [{"text": system}]
    response = bedrock_runtime.converse(**kwargs)
    return response["output"]["message"]["content"][0]["text"]

# Quick test
print(f"Model: {MODEL_ID}")
print("Helper function ready.")

## A. Zero-Shot Prompting

**Zero-shot prompting** means giving the model a task with no examples — the model relies entirely on its pre-training knowledge to produce the answer. This is the simplest and most common prompting approach.

Zero-shot works well when:
- The task is straightforward (summarization, translation, simple classification)
- The model already understands the expected output format
- You do not need highly specific or domain-specialized responses

Zero-shot may struggle when:
- The task requires a very specific output format the model has not seen before
- The classification categories are ambiguous or domain-specific
- High accuracy is required on nuanced or edge-case inputs

### Zero-Shot Classification

We ask the model to classify AWS service descriptions into categories without providing any examples. The model must rely on its pre-training knowledge of AWS services to determine the correct category.

In [ ]:
# Zero-shot classification — no examples provided
descriptions = [
    "A service that lets you run containers without managing servers or clusters.",
    "A fully managed graph database engine optimized for storing billions of relationships.",
    "A service that provides on-demand delivery of compute power, database storage, and other resources.",
    "A managed service for training and deploying machine learning models at scale.",
]

for desc in descriptions:
    prompt = f"""Classify the following AWS service description into exactly one category:
compute, database, storage, networking, ai/ml, security, or analytics.

Description: {desc}

Category:"""
    result = invoke(prompt, temperature=0.0)
    print(f"Description: {desc[:70]}...")
    print(f"Category:    {result.strip()}\n")

### Zero-Shot Summarization

Summarization is one of the strongest zero-shot capabilities of large language models. The model can condense long text into a brief summary without any examples of how to summarize.

In [ ]:
# Zero-shot summarization
text = """Amazon Bedrock is a fully managed service that makes foundation models from
leading AI companies available through a single API. With Bedrock, you can choose
from a wide range of foundation models to find the one best suited for your use
case. Bedrock offers serverless access, so you do not need to manage any
infrastructure. You can customize foundation models with your own data using
techniques like fine-tuning and Retrieval Augmented Generation (RAG). Bedrock also
provides built-in guardrails for responsible AI, including content filtering,
personally identifiable information redaction, and topic-based denial policies.
The service integrates with other AWS services like S3, Lambda, and CloudWatch
for building end-to-end generative AI applications."""

prompt = f"Summarize the following text in exactly 2 sentences:\n\n{text}"
summary = invoke(prompt, temperature=0.3)
print("Summary:")
print(summary)

## B. Few-Shot Prompting

**Few-shot prompting** provides 2-3 examples of the desired input-output behavior before giving the model the actual task. The examples teach the model the expected format, tone, and classification criteria — without any fine-tuning.

Few-shot prompting improves results when:
- The task requires a specific output format the model might not default to
- The classification categories are domain-specific or ambiguous
- You need consistent, predictable responses across many inputs

### How to choose good examples
- **Diverse** — Cover different categories and edge cases, not just the easy ones
- **Representative** — Use examples that look like real production inputs
- **Consistent format** — Every example should follow the exact same input/output structure
- **Balanced** — Include at least one example per category when possible

### Few-Shot Classification

We use the same classification task from Section A, but now we provide 3 examples that demonstrate the expected format. Notice how the examples cover different categories, use consistent formatting, and include a clear separator between each one.

In [ ]:
# Few-shot classification — 3 examples provided
few_shot_prompt = """Classify the AWS service description into exactly one category.
Respond with only the category name, nothing else.

Categories: compute, database, storage, networking, ai/ml, security, analytics

---
Description: A web service that provides resizable compute capacity in the cloud, allowing you to launch virtual servers.
Category: compute

---
Description: A fast, flexible NoSQL database service for single-digit millisecond performance at any scale.
Category: database

---
Description: An object storage service offering industry-leading scalability, data availability, and security.
Category: storage

---
Description: {description}
Category:"""

# Same descriptions as zero-shot for comparison
descriptions = [
    "A service that lets you run containers without managing servers or clusters.",
    "A fully managed graph database engine optimized for storing billions of relationships.",
    "A service that provides on-demand delivery of compute power, database storage, and other resources.",
    "A managed service for training and deploying machine learning models at scale.",
]

for desc in descriptions:
    prompt = few_shot_prompt.format(description=desc)
    result = invoke(prompt, temperature=0.0)
    print(f"Description: {desc[:70]}...")
    print(f"Category:    {result.strip()}\n")

### Comparing Zero-Shot vs Few-Shot

The key difference is **output consistency**. With zero-shot, the model might return "Compute", "compute", "This is a compute service", or other variations. With few-shot examples, the model learns the exact output format you expect — a single lowercase category word — and follows it reliably.

Few-shot is especially valuable for production applications where downstream code needs to parse the model's output programmatically. Consistent formatting means fewer parsing errors.

## C. Chain-of-Thought (CoT) Prompting

**Chain-of-thought prompting** forces the model to show its reasoning step by step before arriving at a final answer. This technique dramatically improves accuracy on tasks that require multi-step reasoning — math, logic, planning, and cost estimation.

The key insight is that large language models generate tokens sequentially. When you ask for just the answer, the model must "jump" to the conclusion. When you ask the model to reason step by step, each intermediate step provides context for the next, reducing errors.

Two common approaches:
- **Zero-shot CoT** — Simply append "Let's think step by step" to the prompt
- **Structured CoT** — Provide a specific reasoning template the model should follow

### Direct Answer (No CoT)

First, let's ask the model a multi-step AWS cost estimation problem with a direct question. The model must perform several calculations to get the right answer.

In [ ]:
# Direct question — no chain-of-thought
problem = """A company runs 3 EC2 instances at $0.10/hour each, 24/7 for a 30-day month.
They also use 500 GB of S3 storage at $0.023 per GB/month and make 1 million
S3 GET requests at $0.0004 per 1000 requests.
What is the total monthly cost?

Answer with just the dollar amount."""

direct_answer = invoke(problem, temperature=0.0)
print("Direct answer (no CoT):")
print(direct_answer)

### With Chain-of-Thought

Now let's ask the same question but instruct the model to reason step by step. This forces it to break the problem into intermediate calculations, making errors easier to catch and correct.

In [ ]:
# Same question with chain-of-thought reasoning
problem_cot = """A company runs 3 EC2 instances at $0.10/hour each, 24/7 for a 30-day month.
They also use 500 GB of S3 storage at $0.023 per GB/month and make 1 million
S3 GET requests at $0.0004 per 1000 requests.
What is the total monthly cost?

Let's think step by step:
1. First, calculate the EC2 cost
2. Then, calculate the S3 storage cost
3. Then, calculate the S3 request cost
4. Finally, add them all together

Show your work for each step, then give the final total."""

cot_answer = invoke(problem_cot, temperature=0.0, max_tokens=1024)
print("Chain-of-thought answer:")
print(cot_answer)

# Verify: 3 * 0.10 * 24 * 30 = $216.00
#         500 * 0.023 = $11.50
#         1,000,000 / 1000 * 0.0004 = $0.40
#         Total = $227.90
print("\n--- Expected answer: $227.90 ---")

### Why CoT Works

Chain-of-thought prompting is effective because:
- **Intermediate tokens provide context** — each reasoning step becomes part of the context for the next step, reducing the chance of compounding errors
- **Errors become visible** — when the model shows its work, you can identify exactly where reasoning breaks down
- **Complex tasks decompose naturally** — multi-step problems that are hard to solve in one jump become manageable as a sequence of simple steps

On the exam, if you see a question about improving accuracy on reasoning or multi-step tasks, **chain-of-thought** is almost always the correct answer.

## D. System Prompts

The Converse API supports a dedicated `system` parameter that sets the model's behavior, persona, and constraints before any user message. System prompts are processed separately from user input, giving them stronger influence over the model's behavior.

System prompts are ideal for:
- **Persona definition** — make the model respond as a specific expert, role, or character
- **Output constraints** — enforce format rules (bullet points only, code only, JSON only)
- **Guardrails** — restrict the model from discussing certain topics or generating certain content
- **Tone and style** — set professional, casual, technical, or educational tone

Below we send the same question to Claude with three different system prompts, demonstrating how each persona produces a fundamentally different response.

In [ ]:
personas = [
    ("AWS Solutions Architect", "You are an AWS Solutions Architect. Answer in bullet points. Maximum 5 points."),
    ("Exam Tutor", "You are a certification exam tutor. Explain concepts with simple analogies and concrete examples."),
    ("Code Reviewer", "You are a Python code expert. Respond only with code, no explanations. Add inline comments."),
]

question = "How do I invoke a Bedrock model using boto3?"

for name, system_prompt in personas:
    print(f"\n{'='*60}")
    print(f"Persona: {name}")
    print(f"{'='*60}")
    result = invoke(question, system=system_prompt)
    print(result)

### System Prompt Best Practices

- **Be specific** — "Answer in bullet points" is better than "Be concise"
- **Set boundaries** — explicitly state what the model should and should not include
- **Combine constraints** — you can layer persona + format + tone in a single system prompt
- **Test edge cases** — users may try to override the system prompt; test with adversarial inputs

In production applications, the system prompt is typically set once per session and applied to all messages in a conversation. It acts as the application's "configuration" for model behavior.

## E. Structured Output Extraction

In production applications, you often need the model to return data in a machine-parseable format — typically JSON. This requires careful prompt engineering to ensure the model returns **only** valid JSON without markdown formatting, explanations, or extra text.

Key techniques for reliable JSON extraction:
- **Set temperature to 0** — eliminates randomness, producing deterministic and consistent output
- **Include the exact JSON schema** in the prompt so the model knows the expected structure
- **Say "return ONLY valid JSON"** — explicitly tell the model not to include any other text
- **Use a system prompt** — reinforce the JSON-only requirement at the system level

### Extract a Single Object

We provide the model with a service description and ask it to extract structured fields into a predefined JSON schema.

In [ ]:
prompt = """Extract the following from this AWS service description and return ONLY valid JSON:
{
  "service_name": "",
  "category": "",
  "key_feature": "",
  "pricing_model": ""
}

Description: Amazon Bedrock is a fully managed service that offers foundation models from leading AI companies through a single API. It supports on-demand and provisioned throughput pricing."""

result = invoke(prompt, temperature=0.0)
print("Raw response:")
print(result)

# Parse and validate
parsed = json.loads(result)
print("\nParsed JSON:")
print(json.dumps(parsed, indent=2))

### Extract a List of Items

This example extracts multiple items from unstructured text into a JSON array. This pattern is common in document processing pipelines where you need to pull structured data from free-text descriptions.

In [ ]:
# Extract a list of items from unstructured text
prompt = """Extract all AWS services mentioned in the text below. For each service,
return a JSON array of objects with "service" and "purpose" keys.
Return ONLY the JSON array, no other text.

Text: Our architecture uses Amazon S3 for storing raw data files, Amazon Bedrock
for running inference on foundation models, AWS Lambda for serverless processing
of incoming requests, and Amazon CloudWatch for monitoring and logging all
API calls across the pipeline."""

result = invoke(
    prompt,
    system="You are a JSON extraction engine. Return only valid JSON, no markdown, no explanation.",
    temperature=0.0
)
print("Raw response:")
print(result)

# Parse and display
services = json.loads(result)
print(f"\nExtracted {len(services)} services:")
for svc in services:
    print(f"  - {svc['service']}: {svc['purpose']}")

### Tips for Reliable JSON Extraction

| Technique | Why It Helps |
|-----------|-------------|
| `temperature=0.0` | Eliminates randomness; model picks the most probable tokens every time |
| Explicit JSON schema in prompt | Model sees exactly what fields and structure you expect |
| "Return ONLY valid JSON" | Prevents the model from wrapping JSON in markdown code blocks or adding explanations |
| System prompt reinforcement | Doubles down on the JSON-only constraint at the system level |
| `json.loads()` validation | Catches any malformed output immediately; wrap in try/except for production use |

In production, always wrap `json.loads()` in a try/except block and implement a retry mechanism for the rare cases where the model returns invalid JSON.

## F. Prompt Chaining

**Prompt chaining** decomposes a complex task into a sequence of simpler steps, where each step's output feeds into the next step's input. Instead of asking the model to do everything in one call, you break the problem into a pipeline of focused operations.

Benefits of prompt chaining:
- **Higher accuracy** — each step handles a simpler, more focused task
- **Debuggability** — you can inspect the output of each step to find where things go wrong
- **Modularity** — you can swap, add, or remove steps without rewriting the entire prompt
- **Targeted parameters** — different steps can use different temperatures (e.g., temperature=0 for classification, temperature=0.7 for creative generation)

Below we chain three Converse calls: **summarize** a document, **classify** the summary, then **recommend** related services based on the classification.

In [ ]:
# Prompt chaining: summarize → classify → recommend

document = """Amazon Bedrock is a fully managed service that makes high-performing foundation
models from leading AI companies available for your use through a unified API. You
can choose from a wide range of foundation models to find the model that is best
suited for your use case. Amazon Bedrock also offers a broad set of capabilities
to build generative AI applications with security, privacy, and responsible AI.
Using Amazon Bedrock, you can easily experiment with and evaluate top foundation
models for your use cases, privately customize them with your data using techniques
such as fine-tuning and Retrieval Augmented Generation (RAG), and build agents that
execute tasks using your enterprise systems and data sources. Since Amazon Bedrock
is serverless, you don't have to manage any infrastructure, and you can securely
integrate and deploy generative AI capabilities into your applications using the
AWS services you are already familiar with."""

# Step 1: Summarize
print("STEP 1 — Summarize")
print("-" * 40)
summary = invoke(f"Summarize this in 2 sentences:\n\n{document}", temperature=0.3)
print(f"{summary}\n")

# Step 2: Classify (using the summary from Step 1)
print("STEP 2 — Classify")
print("-" * 40)
classification = invoke(
    f"Classify this summary into exactly one category (compute/storage/ai-ml/database/networking). "
    f"Respond with only the category name.\n\n{summary}",
    temperature=0.0
)
print(f"Category: {classification.strip()}\n")

# Step 3: Recommend (using both the classification and summary from previous steps)
print("STEP 3 — Recommend")
print("-" * 40)
recommendation = invoke(
    f"Given this {classification.strip()} service summary, suggest 3 complementary AWS services "
    f"that would work well together in a production architecture. For each, explain why in one sentence.\n\n{summary}",
    temperature=0.7
)
print(f"Recommendations:\n{recommendation}")

### Chaining vs Single Prompt

You could attempt the same task in a single prompt — "summarize this document, classify it, and recommend related services." But chaining offers key advantages:

- **Step 2 uses temperature=0** for deterministic classification, while **Step 3 uses temperature=0.7** for creative recommendations — a single prompt cannot use different temperatures per subtask
- If classification is wrong, you can **fix Step 2 independently** without regenerating the summary
- Each step's output is **inspectable** — you can log and debug each stage of the pipeline
- In production, you can **cache intermediate results** — if the document has not changed, skip the summarization step

## Key Takeaways

1. Prompt engineering follows a natural progression: **zero-shot** (simplest) → **few-shot** (add examples) → **chain-of-thought** (add reasoning) → **chaining** (decompose into steps)
2. Set **temperature=0** for deterministic tasks like classification, extraction, and factual Q&A; use higher values for creative generation
3. **System prompts** control model persona and output constraints — they are the primary tool for customizing model behavior in production applications
4. **Structured output** requires explicit format instructions, a JSON schema in the prompt, and temperature=0 for reliability
5. **Prompt chaining** decomposes complex tasks into focused steps, enabling different parameters per step and easier debugging

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Zero-Shot Prompting | Giving the model a task with no examples; relies entirely on pre-training knowledge |
| Few-Shot Prompting | Providing 2-3 input-output examples in the prompt to teach the model the expected format and behavior |
| Chain-of-Thought (CoT) | Instructing the model to show step-by-step reasoning before giving a final answer, improving accuracy on multi-step tasks |
| System Prompt | A dedicated instruction set (via the Converse API `system` parameter) that controls model persona, constraints, and behavior |
| Structured Output | Prompting the model to return data in a machine-parseable format like JSON, using explicit schemas and temperature=0 |
| Prompt Template | A reusable prompt with placeholder variables (e.g., `{description}`) that gets filled in at runtime for consistent formatting |
| Prompt Chaining | Breaking a complex task into a pipeline of sequential model calls, where each step's output feeds into the next step's input |

## Exam Preparation — Common Exam Question Patterns

**Q: Which prompting technique improves accuracy on multi-step reasoning tasks?**
A: Chain-of-thought (CoT) prompting. By forcing the model to show step-by-step reasoning, each intermediate step provides context for the next, reducing errors on math, logic, and planning tasks.

**Q: How do you ensure consistent JSON output from a foundation model?**
A: Use temperature=0 for deterministic output, include the exact JSON schema in the prompt, add an explicit instruction like "return ONLY valid JSON", and optionally reinforce with a system prompt. Always validate the output with `json.loads()`.

**Q: What is the difference between zero-shot and few-shot prompting?**
A: Zero-shot provides no examples — the model relies on pre-training. Few-shot includes 2-3 examples in the prompt that demonstrate the expected input-output format. Few-shot produces more consistent and predictable outputs, especially for classification and extraction tasks.

**Q: How does the Converse API system parameter differ from putting instructions in the user message?**
A: The `system` parameter sets persistent behavioral instructions that are processed separately from user input. System prompts have stronger influence over model behavior and are ideal for persona definition, output constraints, and guardrails. User-message instructions may be overridden by conversational context.

**Q: When should you use prompt chaining instead of a single prompt?**
A: Use prompt chaining when the task involves multiple distinct steps that benefit from different parameters (e.g., temperature=0 for classification, temperature=0.7 for generation), when you need to inspect or debug intermediate results, or when the combined task is too complex for a single prompt to handle reliably.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Bedrock API — Zero-shot (Claude 3.5 Sonnet) | ~2K input + 2K output tokens | ~$0.05 |
| Bedrock API — Few-shot (Claude 3.5 Sonnet) | ~3K input + 1K output tokens | ~$0.04 |
| Bedrock API — Chain-of-thought (Claude 3.5 Sonnet) | ~2K input + 3K output tokens | ~$0.06 |
| Bedrock API — System prompts (Claude 3.5 Sonnet) | ~3K input + 3K output tokens | ~$0.06 |
| Bedrock API — Structured output (Claude 3.5 Sonnet) | ~2K input + 1K output tokens | ~$0.03 |
| Bedrock API — Prompt chaining (Claude 3.5 Sonnet) | ~4K input + 3K output tokens | ~$0.07 |
| **Total** | | **~$0.31** |

All API calls use Claude 3.5 Sonnet via the Converse API. Total cost well under $1.